# Terschelling: Cross-shore time series of shoreline changes as intersection of RTS-DTM with horizontal plane at sea level

## Imports

In [1]:
%load_ext autoreload
%autoreload 2
    
import pickle
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
from scipy.interpolate import griddata

import matplotlib.pyplot as plt

from coastal_data import CD_jarkus_tools, CD_helper_functions, CD_statistics

/home/bene/PhD-git/envs/universe/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## Paths

In [2]:
path_to_data_folder = '/media/bene/Seagate/PhD-data/98_paper2_kf_dtm/'

path_rtsdtm = path_to_data_folder+'12_13_Ters_DEM/output/'
path_input_general = path_to_data_folder+'0_input_general/'
path_altimetry_input = path_to_data_folder+'11_Ters_altimetry/input/'
path_altimetry_output = path_to_data_folder+'11_Ters_altimetry/output/'

## Get RTS-DTM on JARKUS transects

In [3]:
fn_rtsdtm = 'x_state_s.pkl'
x_state_s = pickle.load(open(path_rtsdtm + fn_rtsdtm, 'rb'))

In [4]:
# Jarkus data
jarkus = xr.open_dataset(path_input_general + 'transect_red_with_derivatives.nc')

# Reduce to Terschelling and the time after 1992
idx_alongshore, = np.where(jarkus.areacode == 4)
jarkus_years = pd.to_datetime(jarkus.time).year
idx_time, = np.where((jarkus_years >= 1993) & (jarkus_years <= 2020))
jarkus_years = jarkus_years[idx_time]
jarkus = jarkus.isel(alongshore=idx_alongshore, time=idx_time)

# Remove values coming from more than one source
jarkus['altitude_red'] = xr.where(jarkus.nsources <= 1, jarkus.altitude, np.nan) \
    .assign_attrs({"description": "Values coming \"from more than one source\" are masked out based on variable nsources."})

# De-select transects (same transects as for CASSIE)
fn = 'jarkus_transects_EPSG32631_reduced.geojson'
transects = gpd.read_file(path_input_general + fn).set_index('name')
transects_true = [_ for _ in jarkus.id.values if str(_) in transects.index]
idx_true = np.array([np.where(jarkus.id.values == _)[0][0] for _ in transects_true])
jarkus = jarkus.isel(alongshore=idx_true)

# get lon and lat as griddata likes it
lon_jarkus = jarkus.lon.values.reshape(-1)
lat_jarkus = jarkus.lat.values.reshape(-1)

In [5]:
rts_interp = xr.full_like(jarkus.altitude, fill_value=np.nan).copy()

year_list = [int(col.split('_')[-1]) for col in x_state_s.columns if col.startswith('x_s_')]
jarkus_years = pd.to_datetime(jarkus.time).year.values

for year in year_list:
    mx = x_state_s.geometry.x
    my = x_state_s.geometry.y
    values = x_state_s[f'x_s_{year}']
    interp = griddata(list(zip(mx,my)), values, (rts_interp.lon,rts_interp.lat), 'linear')
    
    year_idx, = np.where(jarkus_years == year)
    rts_interp[dict(time=year_idx)] = interp

jarkus['rts_interp'] = rts_interp

### Remove height difference rts-vali
would be better to consider only differences in the shoreline area  
Bring validation data to SLA -> no need to reduce altimetry to TG then  

In [6]:
diff = jarkus['rts_interp'] - jarkus['altitude_red']
print(f'Mean of all differences (RTS-JARKUS): {diff.mean().round(2).values}, median: {diff.median().round(2).values}')
bias = diff.median().values
jarkus['altitude_red'] += bias

Mean of all differences (RTS-JARKUS): -0.16, median: 0.2


### Plot profiles

## Get sea level

In [7]:
# altimetry
fn = 'tp_plus_ales_epsg28992.csv'
alt = pd.read_csv(path_altimetry_output+fn, index_col='time', parse_dates=['time'])
alt_yearly = alt['sla_temp_av'].groupby(pd.Grouper(freq='YE')).mean()

In [8]:
# tide gauge to get to the same height reference as jarkus
fn = 'psmsl_monthly_RLR_TERS.txt'
tg_psmsl_orig = pd.read_csv(path_altimetry_input+fn, sep=';', header=None, names=['date', 'SSH_RLR[mm]'])

tg_psmsl = pd.DataFrame()
tg_psmsl['date'] = CD_helper_functions.decimal_numbers_to_datetime(tg_psmsl_orig['date'])
tg_psmsl = tg_psmsl.set_index('date')

# translate RLR to NAP (numbers from https://psmsl.org/data/obtaining/rlr.diagrams/236.php)
idx = np.nonzero(tg_psmsl.index.year < 2005)[0]
tg_psmsl.loc[tg_psmsl.index[idx], 'SSH_NAP[cm]'] = ((tg_psmsl_orig.loc[tg_psmsl_orig.index[idx], 'SSH_RLR[mm]'] - 6936) / 10).values

idx = np.nonzero(tg_psmsl.index.year >= 2005)[0]
tg_psmsl.loc[tg_psmsl.index[idx],'SSH_NAP[cm]'] = ((tg_psmsl_orig['SSH_RLR[mm]'][idx] - 6957) / 10).values

# reduce to altimetry time period
tg_psmsl = tg_psmsl.iloc[np.nonzero(tg_psmsl.index.year >= 1992)[0]]

# yearly averages
tg_psmsl_yearly = tg_psmsl.groupby(pd.Grouper(freq='YE')).mean()

## Get intersections

In [9]:
def get_intersection(jarkus, variable, alt_yearly_red, static_profile=False, static_seaLevel=False):
    '''
    Input
    -----
    jarkus - xarray DataSet, 'transect_red_with_derivatives.nc',
             a variable 'rts_interp' (RTS-DTM interpolated to
             the JARKUS transects)
    alt_yearly_red - DataFrame with yearly altimetry data, bias reduced to NAP

    Output
    -----
    '''
    cd_df = pd.DataFrame(index=jarkus.id.values,
                         columns=pd.to_datetime(jarkus.time).year)

    if static_seaLevel == True:
        ssh = 0
        
    for along in jarkus.alongshore:
        transect = jarkus.sel(alongshore=along)[variable]
        
        for time in jarkus.time:
            jarkus_year = pd.to_datetime(time.values).year
            if static_profile == True:
                profile = transect.isel(time=0) # which profile to fix in time
            else:
                profile = transect.sel(time=time)
            
            idx_time, = np.where(alt_yearly_red.index.year == jarkus_year)
            ssh = alt_yearly_red.iloc[idx_time].values
            
            intersections = CD_jarkus_tools.find_intersections(profile, ssh, jarkus.cross_shore)
            DuneTop_prim_x = CD_jarkus_tools.find_primary_dunetop(profile, jarkus.cross_shore)
            cd = CD_jarkus_tools.identify_coastline(intersections, DuneTop_prim_x)
            
            jarkus_id = jarkus.sel(alongshore=along).id.values
            cd_df.loc[int(jarkus_id), jarkus_year] = cd

    return cd_df

In [10]:
# RTS-DTM
# Time-variable profile, time-variable sea level
cd_rts_variable = get_intersection(jarkus, 'rts_interp', alt_yearly, static_profile=False, static_seaLevel=False)
# Time-variable profile, static sea level
cd_rts_static_sl = get_intersection(jarkus, 'rts_interp', alt_yearly, static_profile=False, static_seaLevel=True)
# Static profile, time-variable sea level
cd_rts_static_profile = get_intersection(jarkus, 'rts_interp', alt_yearly, static_profile=True, static_seaLevel=False)

In [11]:
# JARKUS
# Time-variable profile, time-variable sea level
cd_vali_variable = get_intersection(jarkus, 'altitude_red', alt_yearly, static_profile=False, static_seaLevel=False)
# Time-variable profile, static sea level
cd_vali_static_sl = get_intersection(jarkus, 'altitude_red', alt_yearly, static_profile=False, static_seaLevel=True)
# Static profile, time-variable sea level
cd_vali_static_profile = get_intersection(jarkus, 'altitude_red', alt_yearly, static_profile=True, static_seaLevel=False)

## Trends

In [12]:
def get_shoreline_trends(jarkus, cd):
    trend_df = pd.DataFrame(index=jarkus.id.values, columns=['trend', 'Cxx', 'v'])
    x_jarkus = CD_helper_functions.datetime_to_decimal_numbers(pd.to_datetime(jarkus.time.values))
    n_nonNanValues = 5 # minimum number of non-NaN values per time series

    for transect in cd.index:
        ts = cd.loc[transect].values.astype('float')

        if np.all(np.isnan(ts)):
            continue
        nr_non_nan = len(ts) - np.isnan(ts).sum() # number of non nan values
        if nr_non_nan < n_nonNanValues:
            continue
        else:
            # trend_df.loc[transect, 'trend'], _, _ = CD_statistics.compute_trend_with_error(x_jarkus, ts)
            trend_df.loc[transect, 'trend'], trend_df.loc[transect, 'Cxx'], trend_df.loc[transect, 'v'] = CD_statistics.compute_trend_with_error(x_jarkus, ts)
    return trend_df

In [13]:
trend_rts_variable = get_shoreline_trends(jarkus, cd_rts_variable)
trend_rts_static_sl = get_shoreline_trends(jarkus, cd_rts_static_sl)
trend_rts_static_profile = get_shoreline_trends(jarkus, cd_rts_static_profile)

In [14]:
trend_vali_variable = get_shoreline_trends(jarkus, cd_vali_variable)
trend_vali_static_sl = get_shoreline_trends(jarkus, cd_vali_static_sl)
trend_vali_static_profile = get_shoreline_trends(jarkus, cd_vali_static_profile)

### Coastline sections

In [15]:
idx_east = [112, 113, 114, 115, 116, 117, 118, 119, 120,
        121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133,
        134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144]

idx_center = [58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,
         69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,
         82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,
         95,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 107,
        108, 109, 110, 111]

idx_west = [25,  26,  27,
         28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,
         41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,
         54,  55,  56,  57]

In [16]:
def print_results(trend_df, idx_east, idx_center, idx_west):
    trend_all = trend_df.trend
    print(f'Entire coastline: {round(trend_all.mean(),2)} m/year (mean), {round(trend_all.median(),2)} m/year (median)')

    trend_west = trend_df.iloc[idx_west].trend
    print(f'West: {round(trend_west.mean(),2)} m/year (mean), {round(trend_west.median(),2)} m/year (median)')
    
    trend_east = trend_df.iloc[idx_east].trend
    print(f'East: {round(trend_east.mean(),2)} m/year (mean), {round(trend_east.median(),2)} m/year (median)')
    
    trend_center = trend_df.iloc[idx_center].trend
    print(f'Center: {round(trend_center.mean(),2)} m/year (mean), {round(trend_center.median(),2)} m/year (median)')

In [17]:
print(f'Average trends, validation data set:')

print(f'1) Variable profile and variable sea level:\n')
print_results(trend_vali_variable, idx_east, idx_center, idx_west)

print(f'\n\n2) Variable profile and static sea level:\n')
print_results(trend_vali_static_sl, idx_east, idx_center, idx_west)

print(f'\n\n3) Static profile and variable sea level:\n')
print_results(trend_vali_static_profile, idx_east, idx_center, idx_west)

print(f'\n\nAverage trends, RTS-DTM:\n')

print(f'1) Variable profile and variable sea level:\n')
print_results(trend_rts_variable, idx_east, idx_center, idx_west)

print(f'\n\n2) Variable profile and static sea level:\n')
print_results(trend_rts_static_sl, idx_east, idx_center, idx_west)

print(f'\n\n3) Static profile and variable sea level:\n')
print_results(trend_rts_static_profile, idx_east, idx_center, idx_west)

Average trends, validation data set:
1) Variable profile and variable sea level:

Entire coastline: -3.64 m/year (mean), -1.0 m/year (median)
West: -4.93 m/year (mean), -3.67 m/year (median)
East: -17.68 m/year (mean), -15.2 m/year (median)
Center: 4.24 m/year (mean), 3.67 m/year (median)


2) Variable profile and static sea level:

Entire coastline: -3.64 m/year (mean), -1.0 m/year (median)
West: -4.93 m/year (mean), -3.67 m/year (median)
East: -17.68 m/year (mean), -15.2 m/year (median)
Center: 4.24 m/year (mean), 3.67 m/year (median)


3) Static profile and variable sea level:

Entire coastline: -0.49 m/year (mean), -0.29 m/year (median)
West: -0.33 m/year (mean), -0.33 m/year (median)
East: -0.49 m/year (mean), -0.43 m/year (median)
Center: -0.55 m/year (mean), -0.22 m/year (median)


Average trends, RTS-DTM:

1) Variable profile and variable sea level:

Entire coastline: -0.38 m/year (mean), -0.16 m/year (median)
West: -1.2 m/year (mean), -1.03 m/year (median)
East: -1.07 m/year (

The choice of which profile to fix in time doesn't make a lot of difference (tested i=0, 5, and 10), stays in the same magnitude, ~ +- 0.2m/year